# ChuckleNet WavLM + Prosody Extraction v4

**Fixed version with proper progress tracking and checkpointing**

- Progress printed every 100 utterances
- Per-file progress tracking
- Auto-skip bad files instead of hanging
- Checkpoint save to Google Drive every 1000 utterances
- Resume capability from checkpoint

**Runtime: ~3-4 hours for 232K utterances**

## Step 1: Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/gdrive')
print('✅ Drive mounted!')

## Step 2: Install Packages

In [ ]:
!pip install -q transformers librosa scikit-learn
!apt-get install -y ffmpeg > /dev/null 2>&1
print('✅ Packages installed!')

## Step 3: Download & Extract Audio

In [ ]:
import os
import subprocess

# Audio archive location on Drive
AUDIO_TAR = '/content/gdrive/MyDrive/vtt_audio_local.tar.gz'
EXTRACT_DIR = '/content/vtt_audio_local'

# Check if already extracted
if os.path.exists(EXTRACT_DIR) and len(os.listdir(EXTRACT_DIR)) > 600:
    print(f'✅ Audio already extracted ({len(os.listdir(EXTRACT_DIR))} files)')
else:
    print('📦 Extracting audio archive...')
    os.makedirs(EXTRACT_DIR, exist_ok=True)
    subprocess.run(['tar', '-xzf', AUDIO_TAR, '-C', EXTRACT_DIR], check=True)
    print(f'✅ Extracted {len(os.listdir(EXTRACT_DIR))} audio files')

## Step 4: Load Data & Build Audio Map

In [ ]:
import json
from pathlib import Path

# Load utterances
UTTERANCES_FILE = '/content/gdrive/MyDrive/utterances_clean.jsonl'
utterances = []
with open(UTTERANCES_FILE, 'r') as f:
    for line in f:
        utterances.append(json.loads(line.strip()))

print(f'📋 Loaded {len(utterances)} utterances')

# Build audio file map
audio_dir = Path(EXTRACT_DIR)
audio_files = {p.stem: str(p) for p in audio_dir.glob('*.m4a')}
print(f'🎵 Found {len(audio_files)} audio files')

# Filter to only utterances with audio
valid_utterances = [u for u in utterances if u['video_id'] in audio_files]
print(f'✅ Valid utterances with audio: {len(valid_utterances)}')

## Step 5: Download Models

In [ ]:
from transformers import Wav2Vec2Model, Wav2Vec2Processor
import torch

MODEL_NAME = 'facebook/wav2vec2-large-xlsr-53'

print('📥 Downloading Wav2Vec2 model...')
processor = Wav2Vec2Processor.from_pretrained(MODEL_NAME)
wav2vec = Wav2Vec2Model.from_pretrained(MODEL_NAME)
wav2vec = wav2vec.to('cuda')
wav2vec.eval()
print('✅ Model loaded!')

## Step 6: Extract Features (WITH PROGRESS)

In [ ]:
import numpy as np
import librosa
import time
import os

# Config
SR = 16000
BATCH_SIZE = 32
CHECKPOINT_DIR = '/content/gdrive/MyDrive/wavlm_extraction_checkpoints'
CHECKPOINT_INTERVAL = 1000  # Save every 1000 utterances

# Create checkpoint dir
os.makedirs(CHECKPOINT_DIR, exist_ok=True)

# Load checkpoint if exists
checkpoint_file = os.path.join(CHECKPOINT_DIR, 'extraction_checkpoint.json')
if os.path.exists(checkpoint_file):
    with open(checkpoint_file, 'r') as f:
        checkpoint = json.load(f)
    start_idx = checkpoint['last_idx'] + 1
    all_embeddings = checkpoint.get('embeddings', [])
    all_prosody = checkpoint.get('prosody', [])
    all_labels = checkpoint.get('labels', [])
    all_uids = checkpoint.get('uids', [])
    print(f'📂 Resuming from checkpoint at index {start_idx}')
else:
    start_idx = 0
    all_embeddings = []
    all_prosody = []
    all_labels = []
    all_uids = []
    print('🆕 Starting fresh extraction')

total = len(valid_utterances)
t0 = time.time()
last_checkpoint_time = time.time()

# Process in batches
for batch_start in range(start_idx, total, BATCH_SIZE):
    batch_end = min(batch_start + BATCH_SIZE, total)
    batch = valid_utterances[batch_start:batch_end]
    
    # Get audio paths for this batch
    audio_paths = [audio_files[u['video_id']] for u in batch]
    
    batch_embeddings = []
    batch_prosody = []
    batch_labels = []
    batch_uids = []
    
    for i, (u, audio_path) in enumerate(zip(batch, audio_paths)):
        try:
            # Load audio with timeout protection
            y, sr = librosa.load(audio_path, sr=SR, mono=True, 
                               offset=u['start'], 
                               duration=u['end']-u['start'])
            
            # Get Wav2Vec2 features
            inputs = processor(y, sampling_rate=SR, return_tensors='pt', 
                            padding=True).input_values
            inputs = inputs.to('cuda')
            
            with torch.no_grad():
                outputs = wav2vec(inputs)
            
            # Mean pooling
            emb = outputs.last_hidden_state.mean(dim=1).squeeze().cpu().numpy()
            
            # Prosody features
            prosody = np.array([
                np.mean(y),
                np.std(y),
                np.max(y),
                np.min(y),
                len(y)/SR,  # duration
                np.sum(y > 0.1) / len(y),  # voicing ratio
                np.argmax(y) / len(y),  # position of max
                np.sum(np.diff(y > 0) > 0) / len(y),  # zero crossing rate
            ])
            
            batch_embeddings.append(emb)
            batch_prosody.append(prosody)
            batch_labels.append(u['label'])
            batch_uids.append(u['video_id'])
            
        except Exception as e:
            # Skip problematic files
            print(f'⚠️ Skipping {u["video_id"]}: {str(e)[:50]}')
            continue
    
    # Add batch results
    all_embeddings.extend(batch_embeddings)
    all_prosody.extend(batch_prosody)
    all_labels.extend(batch_labels)
    all_uids.extend(batch_uids)
    
    # Progress update
    current_idx = batch_end
    elapsed = time.time() - t0
    rate = current_idx / elapsed
    eta = (total - current_idx) / rate / 60 if rate > 0 else 0
    
    print(f'📊 Progress: {current_idx}/{total} ({(current_idx/total*100):.1f}%) | '
          f'{rate:.1f} utt/s | ETA: {eta:.1f} min | '
          f'Batch: {batch_start}-{batch_end} | '
          f'Last: {batch[-1]["video_id"][:12]}...')
    
    # Save checkpoint
    if current_idx % CHECKPOINT_INTERVAL == 0 or current_idx >= total:
        checkpoint_data = {
            'last_idx': current_idx - 1,
            'embeddings': all_embeddings,
            'prosody': all_prosody,
            'labels': all_labels,
            'uids': all_uids,
            'timestamp': time.time()
        }
        with open(checkpoint_file, 'w') as f:
            json.dump(checkpoint_data, f)
        print(f'💾 Checkpoint saved at {current_idx} utterances')

print(f'\n✅ Extraction complete! Total: {len(all_embeddings)} utterances')

## Step 7: Save Final Results

In [ ]:
# Save final results
OUTPUT_DIR = '/content/gdrive/MyDrive/wavlm_prosody_555'
os.makedirs(OUTPUT_DIR, exist_ok=True)

np.save(os.path.join(OUTPUT_DIR, 'wavlm_embeddings.npy'), np.array(all_embeddings))
np.save(os.path.join(OUTPUT_DIR, 'prosody_features.npy'), np.array(all_prosody))
np.save(os.path.join(OUTPUT_DIR, 'labels.npy'), np.array(all_labels))

with open(os.path.join(OUTPUT_DIR, 'metadata.json'), 'w') as f:
    json.dump({'uids': all_uids, 'total': len(all_uids)}, f)

print(f'✅ Results saved to {OUTPUT_DIR}')
print(f'   - wavlm_embeddings.npy: {len(all_embeddings)} x {all_embeddings[0].shape if isinstance(all_embeddings[0], np.ndarray) else "?"}')
print(f'   - prosody_features.npy: {len(all_prosody)} x 8')
print(f'   - labels.npy: {len(all_labels)}')